# Proximity events in low-altitude traffic

This notebook reads a month of aircraft position reports, keeps daytime reports below 500 ft
AGL, counts pairs of aircraft that came within 1000 ft horizontally and 200 ft vertically
within 5 seconds of each other, aggregates the counts on a hex grid sized to five minutes of
flight, and tests whether any cell has more events than the traffic through it explains. The
outputs are a hex GeoPackage with risk classes, the flagged positions, and a Kepler.gl map with a
time slider.

The data is `data/airspace/aircraft_positions.parquet`, about 1.15 million reports from March
2023 over one metropolitan area, with columns `DATETIME_UTC`, `LATITUDE`, `LONGITUDE`,
`ALTITUDE_AGL_FT`, `GROUND_SPEED_KNTS` and `FLIGHT_UID`. If the file is absent the notebook
writes a synthetic table with the same schema instead. Other schemas are handled by passing a
`PositionSchema` with your column names.

The read and filter run on dask so a year of data works the same way; the event count runs in
overlapping one hour windows across the workers.

In [ ]:
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
warnings.filterwarnings("ignore")

DATA = ROOT / "data" / "airspace"
EXPORTS = ROOT / "exports" / "airspace"
DATA.mkdir(parents=True, exist_ok=True)
EXPORTS.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
print("repo root:", ROOT)

In [ ]:
from datasets.synthetic import synthetic_aircraft_positions
from fortress_gis.compute.cluster import get_dask_client
from fortress_gis.domains import airspace as air
from fortress_gis.features.proximity import ProximityConfig
from fortress_gis.stats.hypothesis import benjamini_hochberg
from fortress_gis.viz.kepler import KeplerMapBuilder, kepler_available

source = DATA / "aircraft_positions.parquet"
if not source.exists():
    synthetic_aircraft_positions(200, points_per_track=120, conflict_pairs=15).to_parquet(source)
    print("wrote synthetic positions")
source

In [ ]:
client = get_dask_client()  # prints the dashboard link; DASK_N_WORKERS etc. override defaults
client

## Load lazily and filter

`load_positions(as_dask=True)` returns a dask-geopandas frame split into partitions of about
250k rows. `AirspaceFilter` keeps reports between 07:00 and 19:00 UTC and between 0 and 500 ft;
a bounding box and a minimum ground speed are optional. Nothing is read until `compute`, which
here is triggered by `len`.

In [ ]:
positions = air.load_positions(source, as_dask=True)
print("partitions:", positions.npartitions)
flt = air.AirspaceFilter(hours=(7, 19), altitude_ft=(0, 500))
filtered = air.filter_positions(positions, flt).compute()
print(f"{len(filtered):,} reports kept, {filtered[air.TRACK].nunique():,} tracks")
filtered.drop(columns="geometry").head()

## Run the pipeline

`run_airspace_pipeline` repeats the filter, then: sizes a hex grid so a cell spans about five
minutes of flight at the median ground speed, counts proximity events per position within
overlapping one hour windows on dask, sums events and reports per hex, and runs three tests.

- A Poisson excess test per hex against the global mean count.
- A Poisson rate test per hex against a tolerance of one event per 10,000 flight hours, with
  exposure measured as summed gaps between consecutive reports of each track (gaps over ten
  minutes dropped, because track identifiers are reused across days).
- A chi-square test of hour of day against hex risk class.

In [ ]:
result = air.run_airspace_pipeline(
    source,
    flt=flt,
    config=ProximityConfig(),
    minutes_per_hex=5,
    use_dask=True,
    window="1h",
)
print("hex radius (m):", round(result.hex_radius_m))
result.summary().round(4)

## Which cells stand out

`cells` has one row per hex with report and event counts, events per thousand reports, both
Poisson p-values and a three level risk class from natural breaks on the event count. The
Benjamini-Hochberg step below shows how many excess flags survive a false discovery rate
correction, which matters when the grid has hundreds of cells.

In [ ]:
cells = result.cells[result.cells["n_points"] > 0]
cols = ["hex_id", "n_points", "event_count", "events_per_1k_reports", "risk_label", "excess_p", "rate_p"]
cells.sort_values("event_count", ascending=False)[cols].head(10).round(6)

In [ ]:
bh = benjamini_hochberg(cells["excess_p"], alpha=0.05)
print("excess p < 0.05:", int((cells["excess_p"] < 0.05).sum()), " after BH:", int(bh["significant"].sum()))
print("rate over tolerance:", int((cells["rate_p"] < 0.05).sum()), "of", len(cells))

## Hour of day

`hourly` counts reports, events and distinct tracks per hour. Events per thousand reports
normalises for traffic volume, so an evening peak in events that matches an evening peak in
traffic is not by itself a finding. The chi-square result says whether the hour and the hex
risk class are independent.

In [ ]:
hourly = result.hourly
fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.bar(hourly["hour"], hourly["reports"], color="#9ecae1", label="reports")
ax1.set_ylabel("reports")
ax2 = ax1.twinx()
ax2.plot(hourly["hour"], hourly["events_per_1k_reports"], color="#d62728", marker="o", label="events per 1k")
ax2.set_ylabel("events per 1,000 reports")
ax1.set_xlabel("hour (UTC)")
ax1.set_title("Traffic and event rate by hour")
plt.show()
hourly

In [ ]:
cs = result.chi_square
if cs is not None:
    print(f"chi2 = {cs.statistic:.1f}, dof = {cs.dof}, p = {cs.p_value:.3g}")
    display(cs.observed)

## Kepler.gl map

The cell below renders the map inline. If the widget shows as blank text, run
`jupyter nbextension enable --py --sys-prefix keplergl` once in the environment and reload the
page; `fortress_gis.viz.kepler.enable_nbextension()` prints the same command. The HTML export in
the last section does not need the extension and opens in any browser.

The position layer is sampled to 100k points for the widget. Kepler reads the timestamp column
and offers a time filter; drag the window to watch the event cells fill in through the day.

In [ ]:
pos = result.positions
sample = pos.sample(min(len(pos), 100_000), random_state=0)
builder = KeplerMapBuilder(title="Proximity events", height=600)
builder.add_layer(result.cells[result.cells["n_points"] > 0], "hex risk", color_field="event_count", opacity=0.55)
builder.add_layer(
    sample,
    "positions",
    color_field="event_count",
    radius=3,
    opacity=0.6,
    time_field=air.TIME,
)
builder.widget() if kepler_available() else print("keplergl not installed")

## Export

The bundle holds the hex grid with risk classes (graduated QML), the filtered positions with
event counts, and `manifest.json` plus `load_in_qgis.py`. The Kepler HTML includes the time
filter.

In [ ]:
paths_out = air.export_artifacts(result, EXPORTS, name="airspace", max_points=150_000)
for k, v in paths_out.items():
    print(f"{k:>10}: {v.relative_to(ROOT)}")
client.close()